# 🧱 Concrete Mix Design as a Constrained Optimization Problem

## ❗ Why this is NOT a simple ML problem

Most ML projects stop at:
> "Given inputs → predict strength"

But real concrete design works in reverse:
> "Given required strength → design a valid mix"

This is an **inverse problem under constraints**, not a prediction task.

---

## 🧠 Core Design Decision

We **do NOT invert the model**.

Instead, we:
1. Generate candidate mixes (search space)
2. Enforce **engineering constraints first** (non-negotiable)
3. Use ML as a **surrogate strength evaluator**
4. Optimize via search (Optuna)

---

## ⚖️ Constraint Hierarchy (CRITICAL)

We enforce constraints in strict order:

1. **Physics (absolute volume, moisture)**
2. **Durability (w/b ratio)**
3. **Code limits (cement, SCM)**
4. **Workability (paste volume)**

👉 If any fails → mix is rejected immediately

This reflects real engineering practice:
> You NEVER trade durability for cost.

---

## 🎯 Objective

Find a mix that:
- Meets strength target
- Satisfies all constraints
- Minimizes cost

---

## ⚠️ Important Assumptions

- Aggregates assumed uniform grading (no sieve curve enforcement)
- Constant air content (2%)
- No slump model (approximated via paste volume)

These are **intentional simplifications**, not oversights.

## 1. imports

In [ ]:
import numpy as np
import pandas as pd
import pickle
import optuna
from functools import lru_cache

## 2. Load Model & Preprocessor

In [ ]:
model_path = '/content/model.pkl'
preprocessor_path = '/content/preprocessor.pkl'

with open(model_path, 'rb') as f:
    model = pickle.load(f)
with open(preprocessor_path, 'rb') as f:
    preprocessor = pickle.load(f)

print(f'✅ Model: {type(model).__name__}')
print(f'✅ Preprocessor: {type(preprocessor).__name__}')

✅ Model: GradientBoostingRegressor
✅ Preprocessor: ColumnTransformer


## 3. Feature Engineering (STRICT CONSISTENCY)
### 🔬 Feature Engineering — Why These Features?

The ML model does not operate on raw inputs alone.

We introduce **domain-aware features**:

### 1. Water-Binder Ratio (w/b)
- Most critical parameter in concrete design
- Directly controls strength and durability

### 2. Log(Age)
- Strength gain is nonlinear over time
- Log transformation captures diminishing returns

### 3. Cement × Age Interaction
- Captures hydration potential over curing duration

### 4. SCM Ratio
- Reflects sustainability vs early strength trade-off

---

## ⚠️ Critical Constraint

These features MUST match training exactly.

Any mismatch → invalid predictions.

In [ ]:
FEATURE_COLS = [
    'Cement','Blast_Furnace_Slag','Fly_Ash','Water',
    'Superplasticizer','Coarse_Aggregate','Fine_Aggregate','Age',
    'Water_Binder_Ratio','Log_Age','Cement_x_Age','SCM_Ratio'
]

def prepare_features(df):
    df = df.copy()
    binder = df['Cement'] + df['Blast_Furnace_Slag'] + df['Fly_Ash']
    df['Water_Binder_Ratio'] = df['Water'] / (binder + 1e-6)
    df['Log_Age'] = np.log(df['Age'] + 1)
    df['Cement_x_Age'] = df['Cement'] * df['Age']
    df['SCM_Ratio'] = (df['Fly_Ash'] + df['Blast_Furnace_Slag']) / (binder + 1e-6)
    return df

### 4. Engineering Constants (REALISTIC)

In [ ]:
SPECIFIC_GRAVITIES = {
    'Cement': 3.15,
    'Fly_Ash': 2.2,
    'Blast_Furnace_Slag': 2.9,
    'Water': 1.0,
    'Fine_Aggregate': 2.65,
    'Coarse_Aggregate': 2.70
}

ABSORPTION = {
    'Fine_Aggregate': 0.02,
    'Coarse_Aggregate': 0.01
}

MOISTURE = {
    'Fine_Aggregate': 0.04,
    'Coarse_Aggregate': 0.02
}

AIR_CONTENT = 0.02

### 5. Code Constraints (STRICT HIERARCHY)

In [ ]:
CONFIG = {
    "max_wb": 0.50,
    "min_cement": 320,
    "max_cement": 450,
    "max_scm": 0.6
}

## 6. Physics Layer (Absolute Volume + Moisture)
### 🧱 Physics Layer — Why Absolute Volume?

Concrete mix design is governed by:

> Total volume = 1 m³

NOT mass balance.

---

## 📌 Key Design Choice

We use **absolute volume method**:

\[
V = \frac{Mass}{Specific\ Gravity \times 1000}
\]

---

## 💧 Moisture Correction (Often Ignored in ML Projects)

Aggregates contain moisture → affects effective water:

- If moisture > absorption → adds water
- If moisture < absorption → removes water

Ignoring this leads to:
❌ incorrect w/b ratio  
❌ wrong strength prediction  

---

## ⚠️ Simplification

- Same SG assumed for aggregates
- Fixed fine/coarse ratio

This keeps system stable while avoiding overfitting to unknown grading curves.

In [ ]:
def apply_physics(mix):
    mix = mix.copy()

    binder = mix['Cement'] + mix['Blast_Furnace_Slag'] + mix['Fly_Ash']

    # Moisture correction
    water_correction = (
        (MOISTURE['Fine_Aggregate'] - ABSORPTION['Fine_Aggregate']) * mix['Fine_Aggregate'] +
        (MOISTURE['Coarse_Aggregate'] - ABSORPTION['Coarse_Aggregate']) * mix['Coarse_Aggregate']
    )

    mix['Water'] += water_correction

    # Absolute volume
    vol = 0
    for mat in ['Cement','Fly_Ash','Blast_Furnace_Slag','Water']:
        vol += mix[mat] / (SPECIFIC_GRAVITIES[mat] * 1000)

    remaining = 1 - (vol + AIR_CONTENT)

    if remaining <= 0:
        return None, ["Volume overflow"]

    # Aggregate split
    fine_ratio = 0.4
    total_agg_mass = remaining * SPECIFIC_GRAVITIES['Fine_Aggregate'] * 1000

    mix['Fine_Aggregate'] = total_agg_mass * fine_ratio
    mix['Coarse_Aggregate'] = total_agg_mass * (1 - fine_ratio)

    # Paste volume
    paste = vol + AIR_CONTENT
    mix['Paste_Volume'] = paste

    return mix, []

## 7. Hard Constraint Filter (NON-NEGOTIABLE)
### 🚫 Hard Constraints — Why We Reject Instead of Adjust

A key design decision:

> We do NOT "fix" bad mixes — we discard them.

---

## ❗ Why?

Auto-adjusting mixes:
- hides engineering violations
- creates unrealistic designs
- breaks trust

---

## 🔴 Non-Negotiable Constraints

1. **Water-Binder Ratio**
   - Durability requirement
   - Cannot be violated

2. **Cement Content**
   - Minimum → strength
   - Maximum → shrinkage control

3. **SCM Ratio**
   - Limits early strength loss

4. **Paste Volume**
   - Proxy for workability

---

## 🧠 Insight

This turns the system into:
> "Feasible region search" instead of naive optimization

In [ ]:
def enforce_constraints(mix):
    violations = []

    binder = mix['Cement'] + mix['Blast_Furnace_Slag'] + mix['Fly_Ash']
    wb = mix['Water'] / binder

    if wb > CONFIG["max_wb"]:
        return None, ["Durability violated (w/b)"]

    if mix['Cement'] < CONFIG["min_cement"]:
        return None, ["Low cement"]

    if mix['Cement'] > CONFIG["max_cement"]:
        return None, ["Excess cement"]

    scm = (mix['Fly_Ash'] + mix['Blast_Furnace_Slag']) / binder
    if scm > CONFIG["max_scm"]:
        return None, ["SCM too high"]

    if not (0.26 <= mix['Paste_Volume'] <= 0.34):
        return None, ["Bad paste volume"]

    return mix, []

### 8. Strength Prediction

In [ ]:
def predict(mix):
    df = prepare_features(pd.DataFrame([mix]))
    X = preprocessor.transform(df[FEATURE_COLS])
    return model.predict(X)[0]

## 9. Cost Function
### 💸 Cost Model — Simplified but Purposeful

We use approximate material costs (₹/kg).

---

## ❗ Why include cost?

Because real-world mix design is:
> "Engineering + Economics"

---

## ⚠️ Limitation

- Static prices
- No regional variation
- No batching/transport cost

---

## 🧠 Insight

Even a rough cost model:
- forces realistic trade-offs
- prevents over-engineered mixes

In [ ]:
COST = {
    "Cement":7,"Fly_Ash":2,"Blast_Furnace_Slag":3,
    "Water":0.05,"Superplasticizer":50,
    "Fine_Aggregate":1.2,"Coarse_Aggregate":1.5
}

def compute_cost(mix):
    return sum(mix[k]*COST[k] for k in COST)

## 10. OPTIMIZATION
### 🎯 Optimization Strategy — Why Optuna?

We are solving:

- Non-linear
- Discontinuous (due to rejection)
- Multi-variable problem

---

## ❌ Why NOT gradient methods?

- No differentiable objective
- Hard constraints break continuity

---

## ✅ Why Optuna?

- Efficient sampling (TPE)
- Handles black-box objective
- Works with rejection-based constraints

---

## ⚖️ Objective Function Design

We optimize:

\[
Loss = |Predicted\ Strength - Target| + \lambda \cdot Cost
\]

Where:
- Strength accuracy is primary
- Cost is secondary

---

## ⚠️ Trade-off

Higher cost penalty:
→ cheaper but less accurate mixes  

Lower cost penalty:
→ accurate but expensive mixes  

We intentionally bias toward **engineering correctness first**.

In [ ]:
def objective(trial, target):

    mix = {
        "Cement": trial.suggest_float("Cement", 300, 450),
        "Blast_Furnace_Slag": trial.suggest_float("Slag", 0, 250),
        "Fly_Ash": trial.suggest_float("FlyAsh", 0, 150),
        "Water": trial.suggest_float("Water", 140, 220),
        "Superplasticizer": trial.suggest_float("SP", 0, 25),
        "Fine_Aggregate": 700,
        "Coarse_Aggregate": 1000,
        "Age": 28
    }

    mix, phys_v = apply_physics(mix)
    if mix is None:
        return 1e6

    mix, cons_v = enforce_constraints(mix)
    if mix is None:
        return 1e6

    strength = predict(mix)
    cost = compute_cost(mix)

    # Objective: strength closeness + cost
    return abs(strength - target) + 0.001 * cost

### 11. Run Optimization

In [ ]:
def recommend(target=40, n_trials=100):

    study = optuna.create_study(direction="minimize")
    study.optimize(lambda t: objective(t, target), n_trials=n_trials)

    best = study.best_params

    mix = {
        "Cement": best["Cement"],
        "Blast_Furnace_Slag": best["Slag"],
        "Fly_Ash": best["FlyAsh"],
        "Water": best["Water"],
        "Superplasticizer": best["SP"],
        "Fine_Aggregate": 700,
        "Coarse_Aggregate": 1000,
        "Age": 28
    }

    mix, _ = apply_physics(mix)
    strength = predict(mix)
    cost = compute_cost(mix)

    return mix, strength, cost

### 12. Decision-Level Explanation
## 🧠 Decision Explanation Layer — Why This Matters

Most ML systems stop at predictions.

This system goes further:
> It explains *why a mix is chosen*

---

## 🔍 What We Explain

1. Strength vs target
2. Durability compliance (w/b)
3. Cement usage reasoning
4. SCM sustainability trade-off
5. Cost implication

---

## 🎯 Goal

Make output usable by:
- Civil engineers
- Site engineers
- Decision-makers

---

## ⚠️ Limitation

Explanation is rule-based, not LLM-generated.

Future improvement:
→ natural language reasoning layer

In [ ]:
def explain(mix, strength, cost, target):

    binder = mix['Cement'] + mix['Blast_Furnace_Slag'] + mix['Fly_Ash']
    wb = mix['Water'] / binder
    scm = (mix['Fly_Ash'] + mix['Blast_Furnace_Slag']) / binder

    explanation = f"""
Decision Summary:

Target Strength: {target} MPa
Achieved Strength: {strength:.2f} MPa

Key Decisions:
- Water-binder ratio = {wb:.2f} → ensures durability compliance
- Cement content = {mix['Cement']:.0f} kg/m³ → balances strength & shrinkage
- SCM usage = {scm:.2f} → improves sustainability

Trade-offs:
- Cost = ₹{cost:.0f}/m³
- Higher durability achieved at moderate cost increase

Engineering Judgment:
This mix prioritizes durability constraints first, then optimizes cost while staying within workability limits.
"""

    return explanation

### 13. Run Example

In [ ]:
mix, strength, cost = recommend(target=40, n_trials=120)

print(mix)
print(explain(mix, strength, cost, 40))

[I 2026-05-02 11:18:27,527] A new study created in memory with name: no-name-6dee1850-40a7-47b7-86b0-296693b88033
[I 2026-05-02 11:18:27,531] Trial 0 finished with value: 1000000.0 and parameters: {'Cement': 417.03194729462217, 'Slag': 103.12349029432782, 'FlyAsh': 129.74184518960277, 'Water': 146.8935367022717, 'SP': 19.218403720175978}. Best is trial 0 with value: 1000000.0.
[I 2026-05-02 11:18:27,534] Trial 1 finished with value: 1000000.0 and parameters: {'Cement': 385.01259245705796, 'Slag': 149.2768131134936, 'FlyAsh': 73.40213911682402, 'Water': 197.8326897974088, 'SP': 16.022579366024562}. Best is trial 0 with value: 1000000.0.
[I 2026-05-02 11:18:27,538] Trial 2 finished with value: 1000000.0 and parameters: {'Cement': 378.1746453512368, 'Slag': 217.58471105094006, 'FlyAsh': 83.38947627276794, 'Water': 193.96391185972846, 'SP': 6.825765916440618}. Best is trial 0 with value: 1000000.0.
[I 2026-05-02 11:18:27,541] Trial 3 finished with value: 1000000.0 and parameters: {'Cement'

{'Cement': 353.858274316756, 'Blast_Furnace_Slag': 59.016076925143224, 'Fly_Ash': 6.66446543902822, 'Water': 176.26989995837684, 'Superplasticizer': 23.021190298685305, 'Fine_Aggregate': 708.0953340821994, 'Coarse_Aggregate': 1062.143001123299, 'Age': 28, 'Paste_Volume': 0.3319855338847175}

Decision Summary:

Target Strength: 40 MPa
Achieved Strength: 39.88 MPa

Key Decisions:
- Water-binder ratio = 0.42 → ensures durability compliance
- Cement content = 354 kg/m³ → balances strength & shrinkage
- SCM usage = 0.16 → improves sustainability

Trade-offs:
- Cost = ₹6270/m³
- Higher durability achieved at moderate cost increase

Engineering Judgment:
This mix prioritizes durability constraints first, then optimizes cost while staying within workability limits.



## 🚀 Final Takeaways

### What this system demonstrates

- ML used as **surrogate model**, not decision-maker
- Engineering constraints dominate optimization
- Explicit trade-off handling (cost vs durability vs strength)
- Real-world physics (volume, moisture) integrated

---

### What this system is NOT

- Not a replacement for lab trials
- Not a certified mix design tool
- Not fully compliant with grading standards

---

### What makes this portfolio strong

This is not:
❌ a regression project  

This is:
✅ a **decision-support system under constraints**

---

### Next Evolution

- Slump prediction model
- Aggregate grading curves (IS 383)
- Multi-objective Pareto front (instead of single solution)
- API + UI deployment

---

> This project bridges the gap between ML and real civil engineering practice.